<a href="https://colab.research.google.com/github/Don4141/ProductPriceAI/blob/main/Modal_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os
os.chdir("/content")
#Clone repo after a new Colab runtime as the previous will be erased
!git clone https://github.com/Don4141/ProductPriceAI.git

#Set repository as working directory
os.chdir("/content/ProductPriceAI")

#Verify
print("Repo exist", os.path.exists("/content/ProductPriceAI"))
print("Current working directory", os.getcwd())

Cloning into 'ProductPriceAI'...
remote: Enumerating objects: 243, done.
remote: Counting objects: 100% (243/243), done.
remote: Compressing objects: 100% (202/202), done.
remote: Total 243 (delta 130), reused 100 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (243/243), 17.95 MiB | 14.09 MiB/s, done.
Resolving deltas: 100% (130/130), done.
Repo exist True
Current working directory /content/ProductPriceAI


In [5]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1 transformers==4.57.6

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [6]:
!pip uninstall -y gradio gradio-client diffusers

Found existing installation: gradio 6.31.0
Uninstalling gradio-6.31.0:
  Successfully uninstalled gradio-6.31.0
Found existing installation: gradio_client 2.7.2
Uninstalling gradio_client-2.7.2:
  Successfully uninstalled gradio_client-2.7.2
Found existing installation: diffusers 0.42.1
Uninstalling diffusers-0.42.1:
  Successfully uninstalled diffusers-0.42.1


In [7]:
#Manage imports
import re
import math
import numpy as np
import torch
import wandb
import transformers
import trl
import peft
import bitsandbytes
import huggingface_hub
from tqdm import tqdm
from pricer.items import Item
from pricer.evaluator import evaluate, Tester
from tqdm.notebook import tqdm
from IPython.display import display, Image
from pathlib import Path
from transformers import AutoTokenizer
from google.colab import userdata
from huggingface_hub import login
import transformers
from peft import LoraConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
import matplotlib.pyplot as plt

print("All LLM training libraries imported successfully!")

All LLM training libraries imported successfully!


In [8]:
#Load API credentials
for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GROQ_API_KEY", "GITHUB_TOKEN",
            "HF_TOKEN", "MODAL_TOKEN_ID", "MODAL_TOKEN_SECRET"]:
  os.environ[key] = userdata.get(key)

In [9]:
#Verify if API keys are loaded
for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GROQ_API_KEY", "GITHUB_TOKEN",
            "HF_TOKEN", "MODAL_TOKEN_ID", "MODAL_TOKEN_SECRET"]:
    print(f"{key}: {'loaded' if os.getenv(key) else 'Missing'}")

OPENAI_API_KEY: loaded
ANTHROPIC_API_KEY: loaded
GROQ_API_KEY: loaded
GITHUB_TOKEN: loaded
HF_TOKEN: loaded
MODAL_TOKEN_ID: loaded
MODAL_TOKEN_SECRET: loaded


In [12]:
#Install modal and ensure modal credentials are successfully loaded
!pip install -q modal

print("Modal Token ID:", bool(os.getenv("MODAL_TOKEN_ID")))
print("Modal Token Secret:", bool(os.getenv("MODAL_TOKEN_SECRET")))

Modal Token ID: True
Modal Token Secret: True


In [13]:
#Verify authentication with Modal
!modal secret list

                                 Secrets                                 
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Name               ┃ Created at           ┃ Created by ┃ Last used at ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ huggingface-secret │ 2026-11-19 10:39 UTC │ don4141    │ -            │
└────────────────────┴──────────────────────┴────────────┴──────────────┘
